In [1]:
import numpy as np

In [2]:
# # create a slinky geometry

# # create a horizontal helix
# import numpy as np

# # create a horizontal helix (axis along x)
# n_nodes = 100
# n_edges = n_nodes - 1
# nodes = []
# edges = []

# R_n = 0.05
# L = 0.2          # total axial length (rename from len)
# n_turns = 4

# theta_max = 2*np.pi * n_turns
# del_theta = theta_max / n_edges
# pitch = L / n_turns  # axial distance per full turn

# for c in range(n_nodes):
#     theta = c * del_theta
#     node_y = R_n * np.cos(theta)
#     node_z = R_n * np.sin(theta)   # sign choice just flips handedness
#     node_x = (pitch / (2*np.pi)) * theta
#     nodes.append([node_x, node_y, node_z])

# for c in range(n_edges):
#     edges.append([c+1, c+2])  # 1-based indexing as you want


In [3]:
import numpy as np

# helix (axis along x) that starts at top and ends at bottom
n_nodes = 100
n_edges = n_nodes - 1
nodes, edges = [], []

R_n = 0.05
L = 0.2
n_turns = 5          # "turn count" used for pairing; actual turns will be n_turns - 0.5

theta_start = 0.0
theta_end = (2*n_turns - 1) * np.pi   # ends at bottom
del_theta = (theta_end - theta_start) / (n_nodes - 1)

# choose x mapping so that x goes from 0 to L over the whole sampled helix
# i.e., L corresponds to theta_end
x_per_rad = L / (theta_end - theta_start)

top_nodes_idx = []
bottom_nodes_idx = []

for c in range(n_nodes):
    theta = theta_start + c * del_theta

    node_y = R_n * np.cos(theta)
    node_z = R_n * np.sin(theta)
    node_x = x_per_rad * (theta - theta_start)

    nodes.append([node_x, node_y, node_z])

    # exact index sets (no floating checks)
    # top targets: 0, 2π, 4π, ..., 2π*(n_turns-1)
    # bottom targets: π, 3π, ..., (2n_turns-1)π
for k in range(n_turns):
    target_top = 2*np.pi*k
    idx_top = int(round((target_top - theta_start)/del_theta))
    top_nodes_idx.append(np.clip(idx_top, 0, n_nodes-1))

    target_bot = np.pi + 2*np.pi*k
    idx_bot = int(round((target_bot - theta_start)/del_theta))
    bottom_nodes_idx.append(np.clip(idx_bot, 0, n_nodes-1))

# make edges (1-based)
for c in range(n_edges):
    edges.append([c+1, c+2])

nodes = np.asarray(nodes, dtype=float)

print("top_nodes_idx:", top_nodes_idx)
print("bottom_nodes_idx:", bottom_nodes_idx)


top_nodes_idx: [np.int64(0), np.int64(22), np.int64(44), np.int64(66), np.int64(88)]
bottom_nodes_idx: [np.int64(11), np.int64(33), np.int64(55), np.int64(77), np.int64(99)]


In [4]:
from pathlib import Path

def write_input_txt(filepath, V, E):
    """
    Write ribbon mesh to custom text format.

    Parameters
    ----------
    filepath : str or Path
        Full path including filename
    V : (N,3) array
        Vertex positions
    E : (E,2) array
        Edge indices
    """
    filepath = Path(filepath)  # converts str → Path safely
    filepath.parent.mkdir(parents=True, exist_ok=True)  # create dirs if needed

    with filepath.open("w") as f:
        f.write("*Nodes\n")
        for v in V:
            f.write(f"{v[0]:.6f}, {v[1]:.6f}, {v[2]:.6f}\n")

        f.write("\n*Edges\n")
        for edge in E:
            f.write(f"{edge[0]}, {edge[1]}\n")


In [5]:
filepath = Path.home() / "GitRepos" / "dismech-python" / \
           "tests" / "resources" / "slinky" / "slinky.txt"

write_input_txt(filepath, nodes, edges)

In [6]:
# save the top and bottom node idx
np.savez(
    "slinky_top_bottom_L.npz",
    top_nodes_idx=np.array(top_nodes_idx),
    bottom_nodes_idx=np.array(bottom_nodes_idx),
    L = L
)


In [7]:
top_nodes = nodes[top_nodes_idx,:]
bottom_nodes = nodes[bottom_nodes_idx,:]

In [8]:
nodes_centerline = 0.5 * (top_nodes + bottom_nodes)   # same length now
# add nodes to start and end to lie exactly at the center of the slinky
nodes_centerline = np.vstack([
    np.array([0.0, 0.0, 0.0]),
    nodes_centerline,
    np.array([L, 0.0, 0.0])
])

print("nodes_centerline:\n", nodes_centerline)

nodes_centerline:
 [[0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [1.11111111e-02 0.00000000e+00 3.06161700e-18]
 [5.55555556e-02 0.00000000e+00 3.06161700e-18]
 [1.00000000e-01 0.00000000e+00 3.06161700e-18]
 [1.44444444e-01 0.00000000e+00 3.06161700e-18]
 [1.88888889e-01 0.00000000e+00 3.06161700e-18]
 [2.00000000e-01 0.00000000e+00 0.00000000e+00]]


In [9]:
filepath_centerline = Path.home() / "GitRepos" / "dismech-python" / \
           "tests" / "resources" / "slinky" / "slinky_centerline.txt"

edges_centerline = [[i,i+1] for i in range(1, len(nodes_centerline))]
write_input_txt(filepath_centerline, nodes_centerline, edges_centerline)